# Day 13 · Interactions — Car or Train

**Question.** Where one 'L' ride could make the trip, how often does Chicago take a ride-hail car
instead, and when? And does that train ride exist for a rider who cannot take the stairs?

**Approach.** One month of ride-hail trips (September 2026), counted by pickup tract, drop-off tract and
hour. A trip **copies the 'L'** when both of its tracts have residents within half a mile (straight line)
of stations on the same line: a one-seat train ride existed. Because a tract is larger than a station's
reach, each trip counts by the expected share: the share of the pickup tract's residents near station A
times the share of the drop-off tract's residents near station B, for the best pair A, B on one line.
The trip is then laid on the track the train would have run. The same test with accessible stations
only tells which copies have a step-free train ride; the rest need stairs.

| Section | What happens |
| --- | --- |
| 1. Setup | Imports, paths, settings |
| 2. Pull | Ride-hail trips by tract pair and hour (cached), all trips by hour, 'L' stops, GTFS |
| 3. Validate | Hours complete, totals, suppressed tracts |
| 4. Transform | Station reach per tract; the copy test; routes along each line |
| 5. Analyze | Copies by segment, hour, line, station; the stairs; the finding |
| 6. QA | Sensitivity checks |
| 7. Export | `plate.json` |

## 1. Setup

In [ ]:
from __future__ import annotations

import json
import logging
import zipfile
from collections import defaultdict
from concurrent.futures import ThreadPoolExecutor
from datetime import date, datetime, timedelta
from pathlib import Path

import numpy as np
import pandas as pd
import requests
import shapely
from shapely.ops import substring

from atlas import gtfs, socrata
from atlas.grid import to_address
from atlas.ledger import Ledger, write_plate

logging.basicConfig(level=logging.INFO, format='%(message)s')
log = logging.getLogger('day13')
DAY = Path('.')
REPO = DAY.resolve().parents[1]
RAW = DAY / 'data' / 'raw'
PROC = DAY / 'data' / 'processed'
OUT = DAY / 'output'
for p in (RAW, PROC, OUT):
    p.mkdir(parents=True, exist_ok=True)

TNP = '6dvr-xwnh'                      # Transportation Network Providers - Trips (2025-)
MONTH = date(2026, 9, 1)               # the latest full month in the dataset
DAYS = 30
REACH_MI = 0.5                         # half a mile, straight line, from a block's internal point
FARE = 2.50                            # CTA full fare, 'L' (O'Hare entry $5 not modelled)
ADA_NOW = {'Racine (Blue Line)', 'Austin (Green Line)'}   # reopened accessible; the portal flag lags (Plate VIII)
LINES = {'red': 'Red', 'blue': 'Blue', 'g': 'Green', 'brn': 'Brown', 'p': 'Purple', 'y': 'Yellow', 'pnk': 'Pink', 'o': 'Orange'}

## 2. Pull

- **Trips:** the portal cannot group a month of trips in one request (it times out), so each clock hour is
  one request: trips with both tracts known, grouped by pickup and drop-off tract, with their count, fare,
  miles and seconds. 720 files, cached in `data/processed/` (about 230 MB, not committed).
- **All trips by hour** (tracts known or not), for the share the tracts cover.
- **Stops:** dated snapshot of CTA's stop list. **Blocks:** 2020 Census blocks (`reference/`).
- **GTFS:** trains leaving their first stop each hour, per line, as the week averages them.

In [ ]:
URL = f'{socrata.BASE_URL}/{TNP}.json'
CACHE = PROC / f'tnp_{MONTH:%Y-%m}'
CACHE.mkdir(exist_ok=True)
SEL = ('pickup_census_tract as p,dropoff_census_tract as d,count(*) as n,'
       'sum(fare) as f,sum(trip_miles) as m,sum(trip_seconds) as s')

def pull_hour(t: datetime) -> int:
    path = CACHE / f'{t:%Y%m%d%H}.json'
    if path.exists():
        return -1
    where = (f"trip_start_timestamp>='{t:%Y-%m-%dT%H:%M:%S}' and trip_start_timestamp<'{t + timedelta(hours=1):%Y-%m-%dT%H:%M:%S}' "
             'and pickup_census_tract is not null and dropoff_census_tract is not null')
    for attempt in range(5):
        try:
            r = requests.get(URL, params={'$select': SEL, '$where': where, '$group': 'p,d', '$limit': 50000}, timeout=120)
            rows = r.json()
            if isinstance(rows, list):
                assert len(rows) < 50000, f'{t}: page full'
                path.write_text(json.dumps(rows))
                return len(rows)
        except (requests.RequestException, ValueError):
            pass
    raise RuntimeError(f'pull failed for {t}')

hours = [datetime(MONTH.year, MONTH.month, 1) + timedelta(hours=i) for i in range(DAYS * 24)]
with ThreadPoolExecutor(6) as ex:
    got = list(ex.map(pull_hour, hours))
log.info('Hours pulled now: %d; cached already: %d', sum(g >= 0 for g in got), sum(g < 0 for g in got))
pulled_tnp = date.fromtimestamp(min(f.stat().st_mtime for f in CACHE.glob('*.json')))

# all trips by hour of day, tracts known or not
all_path = RAW / f'tnp_all_by_hour_{MONTH:%Y-%m}.csv'
if not all_path.exists():
    def day_counts(i):   # one request per day: a month in one request times out
        d = MONTH + timedelta(days=i)
        for attempt in range(5):
            try:
                r = requests.get(URL, params={'$select': 'date_extract_hh(trip_start_timestamp) as hh,count(*) as n',
                                              '$where': f"trip_start_timestamp>='{d}T00:00:00' and trip_start_timestamp<'{d + timedelta(days=1)}T00:00:00'",
                                              '$group': 'hh', '$limit': 50}, timeout=120)
                return pd.DataFrame(r.json()).astype(int)
            except (requests.RequestException, ValueError, KeyError):
                pass
        raise RuntimeError(f'count failed for {d}')
    with ThreadPoolExecutor(6) as ex:
        counts = pd.concat(ex.map(day_counts, range(DAYS)))
    counts.groupby('hh', as_index=False).n.sum().to_csv(all_path, index=False)
all_hour = pd.read_csv(all_path).set_index('hh').n

stops, pulled_stops = socrata.snapshot('8pix-ypme', RAW, 'l_stops')
blocks = pd.read_csv(REPO / 'reference' / 'chicago-blocks-2020.csv', dtype={'geoid': str})
seq = json.loads((REPO / 'reference' / 'cta-line-sequences.json').read_text())
gtfs.RAW_DIR = REPO / gtfs.RAW_DIR
GTFS_ZIP, gtfs_date = gtfs.feed()
print(len(stops), len(blocks), gtfs_date, pulled_tnp)

## 3. Validate

In [ ]:
rows = []
for f in sorted(CACHE.glob('*.json')):
    t = datetime.strptime(f.stem, '%Y%m%d%H')
    for x in json.loads(f.read_text()):
        rows.append((x['p'], x['d'], t.hour, int(x['n']), float(x.get('f', 0)), float(x.get('m', 0)), float(x.get('s', 0))))
T = pd.DataFrame(rows, columns=['p', 'd', 'hh', 'n', 'fare', 'miles', 'secs'])
assert len(list(CACHE.glob('*.json'))) == DAYS * 24, 'missing hours'
# one month summed by tract pair and hour of day
T = T.groupby(['p', 'd', 'hh'], as_index=False)[['n', 'fare', 'miles', 'secs']].sum()
known = T.n.sum()
print(f'trips with both tracts: {known:,} ({known / all_hour.sum():.1%} of {all_hour.sum():,}); '
      f'{known / DAYS:,.0f} a day; tract pairs {T.groupby(["p", "d"]).ngroups:,}')
assert T.p.str.len().eq(11).all() and T.d.str.len().eq(11).all()
print('tracts outside Cook County 17031:', (~T.p.str.startswith('17031')).sum(), (~T.d.str.startswith('17031')).sum())

## 4. Transform

**Stations:** one point per station (`map_id`), the mean of its platforms; its lines; accessible by the
portal flag, with the two corrections of Plate VIII. **Reach:** for every tract, the share of its
residents (2020 blocks) within half a mile of each station.

In [ ]:
st = (stops.assign(lat=stops['location.latitude'].astype(float), lon=stops['location.longitude'].astype(float),
                   ada_b=stops.ada.astype(str).eq('True'))
      .groupby('map_id').agg(name=('station_descriptive_name', 'first'),
                             ada=('ada_b', 'all'), lat=('lat', 'mean'), lon=('lon', 'mean'),
                             **{v: (k, lambda s: s.astype(str).eq('True').any()) for k, v in LINES.items()}))
st.index = st.index.astype(str)
st['short'] = st.name.str.replace(r' \(.*\)$', '', regex=True)
st.loc[st.name.isin(ADA_NOW), 'ada'] = True
assert st.name.isin(ADA_NOW).sum() == 2
st['lines'] = [[L for L in LINES.values() if r[L]] for _, r in st.iterrows()]
st['x'], st['y'] = to_address(st.lat, st.lon)
print(len(st), 'stations,', int((~st.ada).sum()), 'non-accessible')

b = blocks[blocks['pop'] > 0].copy()
b['tract'] = b.geoid.str[:11]
ky = 69.05
kx = ky * np.cos(np.radians(41.88))
dist = np.hypot((b.lat.values[:, None] - st.lat.values[None]) * ky, (b.lon.values[:, None] - st.lon.values[None]) * kx)
near = dist <= REACH_MI
tract_pop = b.groupby('tract')['pop'].sum()
shares = defaultdict(dict)      # tract -> {map_id: share of residents within reach}
for j, m in enumerate(st.index):
    sel = near[:, j]
    if sel.any():
        for t, pop in b['pop'][sel].groupby(b.tract[sel]).sum().items():
            shares[t][m] = pop / tract_pop[t]
print(len(shares), 'of', len(tract_pop), 'tracts have residents within half a mile of a station')

**Routes:** each line's stations in order (`reference/cta-line-sequences.json`); a branch (Green's
Ashland/63rd–Halsted) joins the main line where its track does, at Garfield (found on CTA's GTFS rail
shapes). A route is the list of station-to-station segments between two stations on one line. Each
segment is drawn on the rail shape between its two stations, not as a straight line.

**The copy test:** for a tract pair, the best station pair on any one line (stations differ), with weight
= share near A × share near B. The same test with accessible stations only gives the step-free weight; the
difference is the part of the copy whose train ride needs stairs. Each end gets its own stairs share, so a
station is charged only where its end has no accessible alternative.

In [ ]:
# CTA's GTFS rail shapes, in address units. A round trip (out and back, or round the Loop) is also kept as
# its two halves, split where it turns, so a station is found on the right pass.
RID = {'Red': 'Red', 'Blue': 'Blue', 'Brn': 'Brown', 'G': 'Green', 'Org': 'Orange', 'P': 'Purple', 'Pink': 'Pink', 'Y': 'Yellow'}
rail_trips = pd.read_csv(zipfile.ZipFile(GTFS_ZIP).open('trips.txt'), dtype=str, usecols=['route_id', 'shape_id'])
rail_counts = rail_trips[rail_trips.route_id.isin(RID)].groupby(['route_id', 'shape_id']).size()
rail_counts = rail_counts[rail_counts >= 0.02 * rail_counts.groupby(level=0).transform('sum')]
rail_xy = gtfs.shape_paths(GTFS_ZIP, set(rail_counts.index.get_level_values(1)))
shapes = defaultdict(list)
for rid, sid in rail_counts.index:
    xy = rail_xy[sid][['x', 'y']].to_numpy()
    shapes[RID[rid]].append(shapely.LineString(xy))
    far = int(((xy - xy[0]) ** 2).sum(axis=1).argmax())
    if 2 < far < len(xy) - 3:
        shapes[RID[rid]] += [shapely.LineString(xy[:far + 1]), shapely.LineString(xy[far:])]
assert set(shapes) == set(RID.values()), 'A line has no rail shape'
spt = lambda m: shapely.Point(float(st.loc[m, 'x']), float(st.loc[m, 'y']))

def track_path(L, a, b, near=250):
    # the shortest stretch of one of the line's shapes between stations a and b
    A, B = spt(a), spt(b)
    straight, best = A.distance(B), None
    for ln in shapes[L]:
        if ln.distance(A) > near or ln.distance(B) > near:
            continue
        sa, sb = ln.project(A), ln.project(B)
        sub = substring(ln, min(sa, sb), max(sa, sb))
        if not (0.8 * straight <= sub.length <= 2.2 * straight + 300):
            continue
        if best is None or sub.length < best[0].length:
            best = (sub, sa > sb)
    if best is None:
        return None
    c = list(shapely.simplify(best[0], 8).coords)
    return [[round(x), round(y)] for x, y in (c[::-1] if best[1] else c)]

def join_station(L, main, br):
    # where a branch meets the main line: the first main-line station along its shape past the branch end
    for ln in shapes[L]:
        if ln.distance(spt(br[0])) > 150 or ln.distance(spt(br[-1])) > 150:
            continue
        s0, s1 = ln.project(spt(br[-1])), ln.project(spt(br[0]))
        on = [(abs(ln.project(spt(m)) - s0), m) for m in main
              if ln.distance(spt(m)) < 150 and (ln.project(spt(m)) - s0) * (s0 - s1) > 0]
        if on:
            return min(on)[1]
    raise ValueError(f'{L}: branch {br} has no join on the rail shapes')

on_line = {L: [m for seg in segs for m in seg] for L, segs in seq['lines'].items()}
paths = {}
for L, segs in seq['lines'].items():
    main = segs[0]
    paths[L] = [main]
    for br in segs[1:]:
        paths[L].append(br + [join_station(L, main, br)])
print({L: [st.loc[p[-1], 'short'] for p in ps[1:]] for L, ps in paths.items() if len(ps) > 1})

def route(L, a, b):
    for s in paths[L]:
        if a in s and b in s:
            i, j = s.index(a), s.index(b)
            step = 1 if j > i else -1
            return [(s[k], s[k + step]) for k in range(i, j, step)]
    main = paths[L][0]
    for s in paths[L][1:]:
        for x, y in ((a, b), (b, a)):
            if x in s and y in main:
                r = route(L, x, s[-1]) + route(L, s[-1], y)
                return r if x == a else [(q, p) for p, q in reversed(r)]
    return []

def best(sp, sd, L, acc):
    ok = lambda m: not acc or st.loc[m, 'ada']
    A = [(v, m) for m, v in sp.items() if m in on_line[L] and ok(m)]
    B = [(v, m) for m, v in sd.items() if m in on_line[L] and ok(m)]
    top = (0.0, None, None)
    for v1, a in A:
        for v2, c in B:
            if a != c and v1 * v2 > top[0]:
                top = (v1 * v2, a, c)
    return top

def copy_test(p, d):
    sp, sd = shares.get(p), shares.get(d)
    if not sp or not sd:
        return None
    cand = [(best(sp, sd, L, False), L) for L in on_line]
    (w, a, c), L = max(cand, key=lambda r: r[0][0])
    if w <= 0:
        return None
    w_acc = max(best(sp, sd, LL, True)[0] for LL in on_line)
    # per end: share of that end's reach on line L that has no accessible station
    end = lambda s: 1 - max([v for m, v in s.items() if m in on_line[L] and st.loc[m, 'ada']], default=0) / max(v for m, v in s.items() if m in on_line[L])
    return dict(w=w, L=L, a=a, b=c, stairs=max(0.0, 1 - w_acc / w), ea=end(sp), eb=end(sd))

pairs = T[['p', 'd']].drop_duplicates()
tests = {(p, d): r for p, d in pairs.itertuples(index=False) if (r := copy_test(p, d))}
print(f'{len(tests):,} of {len(pairs):,} tract pairs have a one-seat ride')

## 5. Analyze

In [ ]:
T['w'] = [tests[(p, d)]['w'] if (p, d) in tests else 0.0 for p, d in zip(T.p, T.d)]
T['cp'] = T.n * T.w
T['stairs'] = [T.cp.iat[i] * tests[k]['stairs'] if (k := (T.p.iat[i], T.d.iat[i])) in tests else 0.0 for i in range(len(T))]

hour_trips = (T.groupby('hh').n.sum() / DAYS).reindex(range(24), fill_value=0)
hour_copy = (T.groupby('hh').cp.sum() / DAYS).reindex(range(24), fill_value=0)
trips_day, copy_day, stairs_day = T.n.sum() / DAYS, T.cp.sum() / DAYS, T.stairs.sum() / DAYS
fare_copy = (T.fare * T.w).sum() / T.cp.sum()
miles_copy = (T.miles * T.w).sum() / T.cp.sum()
mins_copy = (T.secs * T.w).sum() / T.cp.sum() / 60

edges = {}         # (line, a, b) in line order -> [24 x copies], [24 x stairs]
by_line = defaultdict(lambda: np.zeros(24)); by_line_st = defaultdict(float)
ends = defaultdict(float); ends_st = defaultdict(float)
sub = T[T.w > 0].groupby(['p', 'd', 'hh'])[['cp', 'stairs']].sum().reset_index()
for r in sub.itertuples(index=False):
    t = tests[(r.p, r.d)]
    v, s = r.cp / DAYS, r.stairs / DAYS
    by_line[t['L']][r.hh] += v
    by_line_st[t['L']] += s
    for tr in (r.p, r.d):          # expected copies with an end within reach of each station of the line
        for m, sh in shares[tr].items():
            if m in on_line[t['L']]:
                ends[m] += v * sh
    if not st.loc[t['a'], 'ada']: ends_st[t['a']] += v * t['ea']
    if not st.loc[t['b'], 'ada']: ends_st[t['b']] += v * t['eb']
    for a, c in route(t['L'], t['a'], t['b']):
        k = (t['L'], a, c) if on_line[t['L']].index(a) < on_line[t['L']].index(c) else (t['L'], c, a)
        e = edges.setdefault(k, [np.zeros(24), np.zeros(24)])
        e[0][r.hh] += v
        e[1][r.hh] += s

line_day = {L: float(h.sum()) for L, h in by_line.items()}
line_stairs = {L: by_line_st[L] / line_day[L] for L in line_day}
share_hour = hour_copy / hour_trips
day_share = copy_day / trips_day
night = [1, 2]                           # 1-3 a.m.
mid = list(range(10, 16))                # 10 a.m.-4 p.m.
night_share = float(hour_copy[night].sum() / hour_trips[night].sum())
mid_share = float(hour_copy[mid].sum() / hour_trips[mid].sum())

# scheduled trains: the week's average day (weekday x5, Saturday, Sunday), by the hour the train leaves its first stop
z = zipfile.ZipFile(GTFS_ZIP)
rd = lambda n, **k: pd.read_csv(z.open(n), dtype=str, **k)
trips_g = rd('trips.txt')
trips_g = trips_g[trips_g.route_id.isin(RID)]
stt = rd('stop_times.txt', usecols=['trip_id', 'departure_time', 'stop_sequence'])
stt = stt[stt.trip_id.isin(set(trips_g.trip_id))]
first = stt.assign(q=stt.stop_sequence.astype(int)).sort_values('q').groupby('trip_id').departure_time.first()
trips_g['hh'] = trips_g.trip_id.map(first).str[:2].astype(int) % 24
week = []
d0 = gtfs_date + timedelta(days=(7 - gtfs_date.weekday()) % 7 or 7)    # the Monday after the feed date
for i in range(7):
    ids = gtfs.service_ids(GTFS_ZIP, d0 + timedelta(days=i))
    week.append(trips_g[trips_g.service_id.isin(ids)].groupby(['route_id', 'hh']).size())
trains = (pd.concat(week, axis=1).fillna(0).sum(axis=1) / 7).unstack(fill_value=0).reindex(columns=range(24), fill_value=0)
trains.index = trains.index.map(RID)
trains_hour = trains.sum()
print('trains per hour:', trains_hour.round(0).astype(int).tolist())

stairs_top = sorted(ends_st.items(), key=lambda kv: -kv[1])[:10]
print(f'{trips_day:,.0f} trips a day with tracts; {copy_day:,.0f} copy the L ({day_share:.1%}); fare {fare_copy:.2f}, '
      f'{miles_copy:.2f} mi, {mins_copy:.1f} min; stairs {stairs_day / copy_day:.1%}')
print('night', f'{night_share:.1%}', 'midday', f'{mid_share:.1%}', '| by hour', share_hour.round(3).tolist())
print({L: round(v) for L, v in sorted(line_day.items(), key=lambda kv: -kv[1])})
print({L: round(v, 3) for L, v in line_stairs.items()})
print([(st.loc[m, 'name'], round(v)) for m, v in stairs_top])

In [ ]:
# The finding, from the numbers above. One rounding per number (the plate and the page share it).
thousands = lambda v: f'{round(v / 1000):,},000'
blue = line_stairs['Blue']
finding = (f'Every day about {thousands(copy_day)} ride-hail trips run between two places beside the same ‘L’ line, '
           f'at about ${fare_copy:.0f} a ride against ${FARE:.2f} by train. '
           f'Between 1 and 3 a.m., when the trains thin out, {"a third" if 0.31 <= night_share < 0.36 else f"{night_share:.0%}"} of all ride-hail trips do. '
           f'On the Blue Line, {blue:.0%} of them copy a train ride that a rider who cannot take the stairs could not make.')
print(finding)

## 6. QA

- Rule sensitivity: stations at least two stops apart; a quarter mile instead of half.
- Tract suppression: the share of all trips the known tracts cover, by hour (suppression is heavier
  where trips are few).

In [ ]:
def two_stops(k):
    t = tests[k]
    return len(route(t['L'], t['a'], t['b'])) >= 2
w2 = np.array([tests[(p, d)]['w'] if (p, d) in tests and two_stops((p, d)) else 0.0 for p, d in zip(T.p, T.d)])
print(f'two stops or more: {(T.n * w2).sum() / T.n.sum():.1%} (all: {day_share:.1%})')
print('known-tract coverage by hour:', (hour_trips * DAYS / all_hour).round(2).tolist())
# every edge's copies add up to the copies times their segment counts
seg_sum = sum(e[0].sum() for e in edges.values())
print(f'segments ridden per copy: {seg_sum / copy_day:.1f}')
# the half-mile edge: give accessible stations 0.6 mi of reach, every other rule unchanged
near6 = dist <= 0.6
shares6 = defaultdict(dict)
for j, m in enumerate(st.index):
    sel = near6[:, j]
    if st.loc[m, 'ada'] and sel.any():
        for t_, pop in b['pop'][sel].groupby(b.tract[sel]).sum().items():
            shares6[t_][m] = pop / tract_pop[t_]
st6 = {}
for k, t in tests.items():
    sp, sd = shares6.get(k[0], {}), shares6.get(k[1], {})
    w6 = max((best(sp, sd, L, True)[0] for L in on_line), default=0) if sp and sd else 0
    st6[k] = max(0.0, 1 - w6 / t['w'])
x6 = pd.Series([T.cp.iat[i] * st6[k] if (k := (T.p.iat[i], T.d.iat[i])) in st6 else 0.0 for i in range(len(T))])
L6 = pd.Series([tests[(p, d)]['L'] if (p, d) in tests else None for p, d in zip(T.p, T.d)])
by6 = (x6.groupby(L6).sum() / T.cp.groupby(L6).sum()).round(3)
print('stairs share with 0.6 mi for accessible stations:', f'{x6.sum() / T.cp.sum():.1%}', by6.to_dict())
assert abs(sum(line_day.values()) - copy_day) < 1

## 7. Export

In [ ]:
pt = lambda m: [round(float(st.loc[m, 'x'])), round(float(st.loc[m, 'y']))]
track = {(L, a, c): track_path(L, a, c) for (L, a, c) in edges}
assert all(track.values()), [k for k, v in track.items() if not v]
edge_rows = [dict(line=L, a=a, b=c, pa=pt(a), pb=pt(c), path=track[(L, a, c)],
                  h=[round(float(v), 1) for v in e[0]], s=[round(float(v), 1) for v in e[1]])
             for (L, a, c), e in sorted(edges.items())]
station_rows = [dict(id=m, name=r['name'], short=r['short'], lines=r['lines'], ada=bool(r.ada), x=round(float(r.x)), y=round(float(r.y)),
                     copies=round(float(ends.get(m, 0))))
                for m, r in st.iterrows()]
# the inset: the Blue Line from the Loop to Logan Square, where the stairs are most of the copies
blue_ids = [m for m in on_line['Blue'] if st.loc[m, 'short'] in ('Clinton', 'Logan Square', 'LaSalle', 'Washington')]
bx, by = st.loc[blue_ids, 'x'], st.loc[blue_ids, 'y']
inset_box = dict(west=float(bx.min()), east=float(st.loc[st.short.eq('Washington/Wabash'), 'x'].max()), south=float(by.min()), north=float(by.max()))
# the track drawn under the bands: shapes carrying at least a tenth of their line's trips (as Plate VII)
main_shapes = rail_counts[rail_counts >= 0.1 * rail_counts.groupby(level=0).transform('sum')]
rail_rows = [dict(line=RID[rid], path=[[round(x), round(y)] for x, y in
                                       shapely.simplify(shapely.LineString(rail_xy[sid][['x', 'y']].to_numpy()), 10).coords])
             for rid, sid in main_shapes.index]
summary = dict(
    month=f'{MONTH:%Y-%m}', days=DAYS, tripsAll=int(all_hour.sum()), tripsKnown=int(known), tripsDay=round(trips_day),
    knownShare=round(float(known / all_hour.sum()), 4),
    copyDay=round(copy_day), copyShare=round(float(day_share), 4), stairsDay=round(stairs_day), stairsShare=round(float(stairs_day / copy_day), 4),
    fareCopy=round(float(fare_copy), 2), milesCopy=round(float(miles_copy), 2), minsCopy=round(float(mins_copy), 1), fare=FARE,
    farePaid=round(float(copy_day * fare_copy)), fareTrain=round(float(copy_day * FARE)),
    nightShare=round(night_share, 4), midShare=round(mid_share, 4), night=night, mid=mid,
    lineDay={L: round(v) for L, v in line_day.items()}, lineStairs={L: round(float(v), 4) for L, v in line_stairs.items()},
    reachMi=REACH_MI, tracts=len(tract_pop), tractsNear=len(shares),
    stations=len(st), nonAccessible=int((~st.ada).sum()),
    twoStopsShare=round(float((T.n * w2).sum() / T.n.sum()), 4),
    stairsWide=dict(reachMi=0.6, share=round(float(x6.sum() / T.cp.sum()), 4), byLine={k: float(v) for k, v in by6.items()}),
    blueStairsOfAll=round(float(by_line_st['Blue'] / stairs_day), 4),
)
plate = dict(
    summary=summary,
    hours=dict(trips=[round(float(v), 1) for v in hour_trips], copies=[round(float(v), 1) for v in hour_copy],
               trains=[round(float(v), 1) for v in trains_hour],
               trainsByLine={L: [round(float(v), 1) for v in trains.loc[L]] for L in trains.index}),
    edges=edge_rows, stations=station_rows, inset=dict(box=inset_box), rail=rail_rows,
    sources=dict(tnp=f'https://data.cityofchicago.org/d/{TNP}', stops='https://data.cityofchicago.org/d/8pix-ypme',
                 gtfs='https://www.transitchicago.com/developers/gtfs/', fares='https://www.transitchicago.com/fares/'),
    gtfsDate=gtfs_date.isoformat(), pulledStops=pulled_stops.isoformat(),
)
ledger = Ledger(
    source='City of Chicago, Transportation Network Providers - Trips (2025-)', dataset_id=TNP,
    url=f'https://data.cityofchicago.org/d/{TNP}', pulled=pulled_tnp, units='trips per day, September 2026',
    method=['trips with both census tracts, by pickup tract, drop-off tract and hour',
            'a copy: both tracts have residents within half a mile of stations on one line (expected share)'],
    caveats=[f'{1 - summary["knownShare"]:.0%} of trips have a tract withheld and are left out',
             'fares rounded to $2.50 by the City; tips and charges not included'],
)
write_plate(OUT / 'plate.json', day=13, ledger=ledger, finding=finding, payload=plate)
print('plate.json', round((OUT / 'plate.json').stat().st_size / 1024), 'KB')